In [ ]:
#### Preprocessing and Splitting Data for Regression Models

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

#### 1. Load data
df = pd.read_csv('cleaned_charging_records.csv')
df['StartDatetime'] = pd.to_datetime(df['StartDatetime'])

#### 2. Features
df['StartHour'] = df['StartDatetime'].dt.hour
df['DayOfWeek'] = df['StartDatetime'].dt.dayofweek
df['Month'] = df['StartDatetime'].dt.month
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

df['Hour_sin'] = np.sin(2 * np.pi * df['StartHour'] / 24.0)
df['Hour_cos'] = np.cos(2 * np.pi * df['StartHour'] / 24.0)
df['Day_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7.0)
df['Day_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7.0)

#### 3. Drop leakage & unneeded columns
leakage_cols = ['Duration', 'EndDatetime', 'EndDay', 'EndTime']
unneeded_cols = ['UserID', 'StartDatetime', 'StartDay', 'StartTime', 'StartHour', 'DayOfWeek']
df_model = df.drop(columns=leakage_cols + unneeded_cols)

#### 4. Encode & Split
df_encoded = pd.get_dummies(df_model, columns=['Location', 'ChargerCompany', 'ChargerType'], drop_first=True)
X = df_encoded.drop(columns=['Demand'])
y = df_encoded['Demand']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Regression - Charging Demand Prediciton
## Linear Regression , Decision Tree, Random Forest, Gradiant Boosting, XGBoost and LightGBM 
## Evaluation matrix - MAE, MSE,RMSE and R2

In [19]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Storage dictionary to hold evaluation metrics for all 6 models
results_dict = {}

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    # Store results in the dictionary
    results_dict[name] = {
        'MAE (kWh)': round(mae, 3),
        'MSE': round(mse, 3),
        'RMSE (kWh)': round(rmse, 3),
        'R² Score': round(r2, 4)
    }
    
    # Print metrics cleanly for immediate inspection
    print(f"--- {name} ---")
    print(f"MAE:  {mae:.2f} kWh")
    print(f"MSE:  {mse:.2f}")
    print(f"RMSE: {rmse:.2f} kWh")
    print(f"R²:   {r2:.4f}\n")


In [4]:
### Linear Regression - train the model 
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

evaluate_model("LinearRegeression", y_test, y_pred_lr)

--- LinearRegeression ---
MAE:  9.67 kWh
MSE:  161.31
RMSE: 12.70 kWh
R²:   0.1038



In [8]:
### Decision Tree Regressor 

from sklearn.tree import DecisionTreeRegressor
dt_model = DecisionTreeRegressor(max_depth= 8, random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict (X_test)

evaluate_model("Decision Tree", y_test,y_pred_dt)


--- Decision Tree ---
MAE:  9.13 kWh
MSE:  148.93
RMSE: 12.20 kWh
R²:   0.1726



In [11]:
## RAndom Forest 
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators= 100, max_depth= 12 , random_state=42,n_jobs=-1 )
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
evaluate_model("Random Forest", y_test, y_pred_rf)

--- Random Forest ---
MAE:  8.61 kWh
MSE:  134.41
RMSE: 11.59 kWh
R²:   0.2532



In [20]:
### Gradient Boosting 
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor( n_estimators = 100, max_depth =5,learning_rate = 0.1, random_state = 42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)
evaluate_model("Gradient Boosting", y_test, y_pred_gb)

--- Gradient Boosting ---
MAE:  8.86 kWh
MSE:  139.74
RMSE: 11.82 kWh
R²:   0.2236



In [ ]:
### XGBoost Regressor
import xgboost as xgb


xgb_model = xgb.XGBRegressor(n_estimators =100, max_depth =5, learning_rate =0.1, reg_alpha = 0.1
            ,reg_lambda =1.0,n_jobs= -1, random_state = 42 )

xgb_model.fit(X_train,y_train)
y_pred_xgb = xgb_model.predict(X_test)
evaluate_model("XGBoost", y_test,y_pred_xgb)


--- XGBoost ---
MAE:  8.94 kWh
MSE:  141.16
RMSE: 11.88 kWh
R²:   0.2157



In [ ]:
### LightGBM Regressor
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(n_estimators=100,learning_rate=0.1,max_depth=6,
    num_leaves=31,random_state=42,n_jobs=-1)

lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)
evaluate_model("LightGBM", y_test, y_pred_lgb)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001770 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 353
[LightGBM] [Info] Number of data points in the train set: 57686, number of used features: 22
[LightGBM] [Info] Start training from score 17.592737
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
--- LightGBM ---
MAE:  8.83 kWh
MSE:  138.62
RMSE: 11.77 kWh
R²:   0.2298



In [27]:
# 1. Reset dictionary
results_dict = {}

# 2. Evaluate all 6 models from memory
evaluate_model("Linear Regression", y_test, lr_model.predict(X_test))
evaluate_model("Decision Tree", y_test, dt_model.predict(X_test))
evaluate_model("Random Forest", y_test, rf_model.predict(X_test))
evaluate_model("Gradient Boosting", y_test, gb_model.predict(X_test))
evaluate_model("XGBoost", y_test, xgb_model.predict(X_test))
evaluate_model("LightGBM", y_test, lgb_model.predict(X_test))

# 3. Build and display the complete 6-model comparison table
results_df = pd.DataFrame(results_dict).T.sort_values(by="R² Score", ascending=False)
results_df

--- Linear Regression ---
MAE:  9.67 kWh
MSE:  161.31
RMSE: 12.70 kWh
R²:   0.1038

--- Decision Tree ---
MAE:  9.13 kWh
MSE:  148.93
RMSE: 12.20 kWh
R²:   0.1726

--- Random Forest ---
MAE:  8.61 kWh
MSE:  134.41
RMSE: 11.59 kWh
R²:   0.2532

--- Gradient Boosting ---
MAE:  8.86 kWh
MSE:  139.74
RMSE: 11.82 kWh
R²:   0.2236

--- XGBoost ---
MAE:  8.94 kWh
MSE:  141.16
RMSE: 11.88 kWh
R²:   0.2157

--- LightGBM ---
MAE:  8.83 kWh
MSE:  138.62
RMSE: 11.77 kWh
R²:   0.2298



,MAE (kWh),MSE,RMSE (kWh),R² Score
Random Forest,8.610,134.406,11.593,0.2532
LightGBM,8.830,138.619,11.774,0.2298
Gradient Boosting,8.864,139.744,11.821,0.2236
XGBoost,8.937,141.163,11.881,0.2157
Decision Tree,9.127,148.927,12.204,0.1726
Linear Regression,9.671,161.307,12.701,0.1038


In [28]:
import pickle

# 1. Save the trained Random Forest model
with open('best_rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# 2. Save the exact list of feature columns
model_features = list(X.columns)
with open('model_features.pkl', 'wb') as f:
    pickle.dump(model_features, f)

print("Files saved successfully using pickle!")

Files saved successfully using pickle!
